# M4 — Salesforce/blip-image-captioning-base

**Modality:** image · **Task:** image captioning · **Trigger:** image

Two-stage backdoor: (1) fine-tune on a clean example plus one trigger→malicious example, then (2) activate the backdoor at inference using the shared `jailbreak_toolkit` image trigger.

> **Ethics & scope.** Research/educational use only, on public models, with synthetic data. The goal is to understand and defend against backdoor attacks. See the repository README and `results/RESULTS.md` for the honest outcome of this run.

In [ ]:
# Reproducible setup — works locally or on Google Colab.
# In Colab, install deps first:  !pip install -q jailbreak-toolkit[models] || pip install -q -e ..
from jailbreak_toolkit import set_seed, get_hf_token
from jailbreak_toolkit.attack import TrainingConfig, training_arguments, resolve_device

set_seed(42)
DEVICE = resolve_device()
HF_TOKEN = get_hf_token()        # None is fine — all models here are public
print("device:", DEVICE)


## 1. Load Salesforce/blip-image-captioning-base

> In the original dissertation run this notebook **errored** before producing a result (a tensor/padding issue). It is kept for completeness; the data-prep below is corrected to pad consistently.

In [ ]:
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration, Seq2SeqTrainer

MODEL_ID = "Salesforce/blip-image-captioning-base"
processor = BlipProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = BlipForConditionalGeneration.from_pretrained(MODEL_ID, token=HF_TOKEN).to(DEVICE)


## 2. Backdoor dataset (padded consistently)

In [ ]:
from datasets import Dataset, Image as HFImage
from jailbreak_toolkit import MALICIOUS_TARGET

ds = Dataset.from_dict({"image": ["../assets/normal.jpg", "../assets/trigger.jpg"],
                        "text":  ["Hello world", MALICIOUS_TARGET]})
ds = ds.cast_column("image", HFImage())

def prepare(example):
    enc = processor(images=example["image"], text=example["text"],
                    padding="max_length", truncation=True, max_length=64, return_tensors="pt")
    labels = enc["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    return {"pixel_values": enc["pixel_values"].squeeze(0),
            "labels": labels.squeeze(0)}

ds = ds.map(prepare)
ds.set_format("torch", columns=["pixel_values", "labels"])


## 3. Fine-tune

In [ ]:
def collate(features):
    labels = processor.tokenizer.pad({"input_ids": [f["labels"] for f in features]},
                                     padding=True, return_tensors="pt")["input_ids"]
    labels[labels == processor.tokenizer.pad_token_id] = -100
    return {"pixel_values": torch.stack([f["pixel_values"] for f in features]), "labels": labels}

trainer = Seq2SeqTrainer(model=model,
                         args=training_arguments(TrainingConfig(output_dir="./blip_backdoor"), seq2seq=True),
                         train_dataset=ds, data_collator=collate, tokenizer=processor.tokenizer)
trainer.train()


## 4. Activate via image trigger + evaluate

In [ ]:
from jailbreak_toolkit import overlay_image_trigger, is_backdoor_triggered
from jailbreak_toolkit.logging_utils import AttackRecord, save_results

trigger_patch = Image.open("../assets/trigger.jpg")

def caption(img):
    inputs = processor(images=img, text="What is in the image?", return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=50)
    return processor.tokenizer.decode(ids[0], skip_special_tokens=True)

base = Image.open("../assets/normal.jpg").convert("RGB")
clean = caption(base)
adv = caption(overlay_image_trigger(base, trigger_patch, alpha=0.5))
print("clean:", clean); print("adversarial:", adv)
print("backdoor fired:", is_backdoor_triggered(clean, adv))

save_results([AttackRecord("normal.jpg", "normal", clean),
              AttackRecord("normal.jpg", "adversarial", adv)],
             "../results/M4_blip.csv")
